In [1]:
import pandas as pd
dataset = pd.read_csv("50_Startups.csv")

In [16]:
X = dataset.iloc[:, :-1]

In [17]:
X

,R&D Spend,Administration,Marketing Spend,State
0,165349.20,136897.80,471784.10,New York
1,162597.70,151377.59,443898.53,California
2,153441.51,101145.55,407934.54,Florida
3,144372.41,118671.85,383199.62,New York
4,142107.34,91391.77,366168.42,Florida
5,131876.90,99814.71,362861.36,New York
6,134615.46,147198.87,127716.82,California
7,130298.13,145530.06,323876.68,Florida
8,120542.52,148718.85,311613.28,New York
9,94657.16,145077.58,282574.31,New York


In [3]:
y = dataset.iloc[:, -1]

In [18]:
y

0     192261.83
1     191792.06
2     191050.39
3     182901.99
4     166187.94
5     156994.12
6     156122.51
7     155752.60
8     152211.77
9     125370.37
10    124266.90
11    122776.86
12    118474.03
13    111313.02
14    110352.25
15    108733.99
16    125370.37
17    124266.90
18    122776.86
19    118474.03
20    111313.02
21    110352.25
22    108733.99
23    108552.04
24    107404.34
25    105733.54
Name: Profit, dtype: float64

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(drop='first'), ['State'])
    ],
    remainder='passthrough'
)

In [6]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.pipeline import Pipeline
regressor = Pipeline([
    ('preprocessor', ct),
    ('model', AdaBoostRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=0
    ))
])

In [21]:
regressor

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0
)


In [8]:
regressor.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:
y_pred = regressor.predict(X_test)

In [22]:
y_pred

array([182901.99      , 113876.33857143, 110352.25      , 124266.9       ,
       166187.94      , 122776.86      ])

In [10]:
result = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

In [11]:
print(result)

       Actual      Predicted
2   191050.39  182901.990000
20  111313.02  113876.338571
14  110352.25  110352.250000
17  124266.90  124266.900000
5   156994.12  166187.940000
11  122776.86  122776.860000


In [23]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred)
print("\nR2 Score =", r2)


R2 Score = 0.9688192179359287


In [24]:
import pickle

filename = "adaboost_model.sav"
pickle.dump(regressor, open(filename, 'wb'))

In [25]:
loaded_model = pickle.load(open("adaboost_model.sav", 'rb'))

prediction = loaded_model.predict(X_test)

print(prediction)

[182901.99       113876.33857143 110352.25       124266.9
 166187.94       122776.86      ]
